In [1]:
import os
import pandas as pd
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

print("All Libraries Imported Successfully!")

All Libraries Imported Successfully!


In [2]:
pdf_path = "../dataset/sample.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

with open("../dataset/extracted_text.txt", "w", encoding="utf-8") as file:
    file.write(text)

print("PDF Text Extracted Successfully!")
print("Total Characters:", len(text))

PDF Text Extracted Successfully!
Total Characters: 40855


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(text)

print("Text Chunking Completed!")
print("Total Chunks:", len(chunks))

Text Chunking Completed!
Total Chunks: 103


In [4]:
chunks_df = pd.DataFrame({
    "Chunk_ID": range(1, len(chunks) + 1),
    "Text": chunks
})

chunks_df.to_csv("../dataset/chunks.csv", index=False)

print("Chunks Saved Successfully!")
print("Total Records:", len(chunks_df))

Chunks Saved Successfully!
Total Records: 103


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings Generated Successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings Generated Successfully!


In [6]:
vector_store = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    collection_name="pdf_qa_documents",
    persist_directory="../chroma_db"
)

print("Documents Stored in ChromaDB Successfully!")
print("Total Documents:", vector_store._collection.count())

Documents Stored in ChromaDB Successfully!
Total Documents: 206


In [7]:
llm = OllamaLLM(
    model="gemma3:1b",
    temperature=0
)

print("Ollama LLM Loaded Successfully!")

Ollama LLM Loaded Successfully!


In [8]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context given below.

If the answer is not available in the context, say:
"The answer is not available in the PDF."

Context:
{context}

Question:
{question}

Answer:
""")

print("Question Answering Prompt Created Successfully!")

Question Answering Prompt Created Successfully!


In [9]:
query = "Which Python framework enables developers to quickly build web applications for machine learning and LLM projects?"

print("User Query:", query)

User Query: Which Python framework enables developers to quickly build web applications for machine learning and LLM projects?


In [10]:
results = vector_store.similarity_search(query, k=5)

retrieved_context = "\n\n".join(
    document.page_content for document in results
)

print("Retrieved Context:")
print(retrieved_context)

Retrieved Context:
through a web browser.  
• To prepare the complete application for the final capstone project in Week 15.  
Task Explanation 
Most AI applications are deployed with graphical user interfaces so that end users can interact with them 
easily. Streamlit is a Python framework that enables developers to quickly build web applications for 
machine learning and LLM projects. 
In this task, students will convert the PDF Question Answering System developed in Week 13 into an

through a web browser.  
• To prepare the complete application for the final capstone project in Week 15.  
Task Explanation 
Most AI applications are deployed with graphical user interfaces so that end users can interact with them 
easily. Streamlit is a Python framework that enables developers to quickly build web applications for 
machine learning and LLM projects. 
In this task, students will convert the PDF Question Answering System developed in Week 13 into an

on the web interface. 
Display the re

In [11]:
formatted_prompt = prompt.format(
    context=retrieved_context,
    question=query
)

answer = llm.invoke(formatted_prompt)

print("Generated Answer:")
print(answer)

Generated Answer:
Streamlit


In [12]:
answers_df = pd.DataFrame([{
    "Question": query,
    "Retrieved_Context": retrieved_context,
    "Answer": answer
}])

answers_df.to_csv("../dataset/answers.csv", index=False)

print("Answer Saved Successfully!")
print(answers_df)

Answer Saved Successfully!
                                            Question  \
0  Which Python framework enables developers to q...   

                                   Retrieved_Context     Answer  
0  through a web browser.  \n• To prepare the com...  Streamlit  


# HOME ACTIVITY
## PDF Question Answering System using RAG

### Objective
To build a PDF Question Answering System using LangChain, ChromaDB, Sentence Transformers and Ollama.

### PDF Details
- Topic: Introduction to Artificial Intelligence
- Total Pages: 17
- Chunk Size: 500
- Chunk Overlap: 100
- Embedding Model: all-MiniLM-L6-v2
- LLM: gemma3:1b

In [13]:
from pypdf import PdfReader

pdf_path = "../dataset/home_activity.pdf"
reader = PdfReader(pdf_path)

home_text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        home_text += page_text + "\n"

print("Home Activity PDF Loaded Successfully!")
print("Total Pages:", len(reader.pages))
print("Total Characters:", len(home_text))

Home Activity PDF Loaded Successfully!
Total Pages: 17
Total Characters: 22468


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunk Size Experiments

splitter_300 = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
home_chunks_300 = splitter_300.split_text(home_text)

splitter_500 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
home_chunks_500 = splitter_500.split_text(home_text)

splitter_800 = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)
home_chunks_800 = splitter_800.split_text(home_text)

print("300 Size:", len(home_chunks_300), "Chunks")
print("500 Size:", len(home_chunks_500), "Chunks")
print("800 Size:", len(home_chunks_800), "Chunks")

300 Size: 92 Chunks
500 Size: 57 Chunks
800 Size: 35 Chunks


In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

home_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Home Activity Embeddings Loaded Successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Home Activity Embeddings Loaded Successfully!


In [16]:
from langchain_chroma import Chroma

home_vector_store = Chroma(
    collection_name="home_activity_documents",
    embedding_function=home_embeddings,
    persist_directory="../home_chroma_db"
)

print("Home Activity ChromaDB Loaded Successfully!")
print("Total Documents:", home_vector_store._collection.count())

Home Activity ChromaDB Loaded Successfully!
Total Documents: 57


In [17]:
import pandas as pd

queries_df = pd.read_csv("../dataset/queries.csv")

print("15 Questions Loaded Successfully!")
print(queries_df)
print("Total Questions:", len(queries_df))

15 Questions Loaded Successfully!
    Query_ID                                           Question
0          1                   What is Artificial Intelligence?
1          2  What are the main objectives of Artificial Int...
2          3                What are the characteristics of AI?
3          4         What are the limitations and ethics of AI?
4          5                     What are AI search strategies?
5          6                       What is a state space in AI?
6          7  What are the different types of knowledge repr...
7          8                       What is uncertain knowledge?
8          9                        What is Bayesian reasoning?
9         10                          What is machine learning?
10        11      What are the three types of machine learning?
11        12                          What is an expert system?
12        13                What are rule-based expert systems?
13        14               What is Natural Language Processing?
14    

In [18]:
# Retrieve Top-5 Chunks for Each Question

retrieved_chunks_data = []

for index, row in queries_df.iterrows():

    question = row["Question"]

    results = home_vector_store.similarity_search(
        question,
        k=5
    )

    print("\nQuestion:", question)
    print("Retrieved Chunks:", len(results))

    for chunk_number, result in enumerate(results, start=1):
        retrieved_chunks_data.append({
            "Query_ID": row["Query_ID"],
            "Question": question,
            "Chunk_Number": chunk_number,
            "Retrieved_Chunk": result.page_content
        })

retrieved_df = pd.DataFrame(retrieved_chunks_data)

retrieved_df.to_csv(
    "../dataset/retrieved_chunks.csv",
    index=False
)

print("\nTop-5 Retrieval Completed!")
print("Total Retrieved Chunks:", len(retrieved_df))


Question: What is Artificial Intelligence?
Retrieved Chunks: 5

Question: What are the main objectives of Artificial Intelligence?
Retrieved Chunks: 5

Question: What are the characteristics of AI?
Retrieved Chunks: 5

Question: What are the limitations and ethics of AI?
Retrieved Chunks: 5

Question: What are AI search strategies?
Retrieved Chunks: 5

Question: What is a state space in AI?
Retrieved Chunks: 5

Question: What are the different types of knowledge representation?
Retrieved Chunks: 5

Question: What is uncertain knowledge?
Retrieved Chunks: 5

Question: What is Bayesian reasoning?
Retrieved Chunks: 5

Question: What is machine learning?
Retrieved Chunks: 5

Question: What are the three types of machine learning?
Retrieved Chunks: 5

Question: What is an expert system?
Retrieved Chunks: 5

Question: What are rule-based expert systems?
Retrieved Chunks: 5

Question: What is Natural Language Processing?
Retrieved Chunks: 5

Question: What are intelligent agents and PEAS?
Re

In [19]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

# Load Ollama LLM
llm = OllamaLLM(
    model="gemma3:1b",
    temperature=0
)

# RAG Prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the given context.

If the answer is not available in the context,
say "The answer is not available in the PDF."

Context:
{context}

Question:
{question}

Answer:
""")

print("Ollama LLM Loaded Successfully!")

Ollama LLM Loaded Successfully!


In [20]:
# Compare Answers With and Without RAG

answers_data = []

rag_chain = prompt | llm

for index, row in queries_df.iterrows():

    question = row["Question"]

    # Without RAG
    without_rag_answer = llm.invoke(question)

    # Retrieve Top-5 Chunks
    results = home_vector_store.similarity_search(
        question,
        k=5
    )

    context = "\n\n".join(
        result.page_content for result in results
    )

    # With RAG
    rag_answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    answers_data.append({
        "Query_ID": row["Query_ID"],
        "Question": question,
        "Without_RAG_Answer": without_rag_answer,
        "With_RAG_Answer": rag_answer
    })

    print("\nQuestion:", question)
    print("Without RAG:", without_rag_answer)
    print("With RAG:", rag_answer)

print("\nAnswer Comparison Completed!")


Question: What is Artificial Intelligence?
Without RAG: Okay, let's break down what Artificial Intelligence (AI) is. It's a really broad and rapidly evolving field, but here's a breakdown of the key concepts:

**1. What is Artificial Intelligence?**

At its core, Artificial Intelligence is about creating machines that can perform tasks that typically require human intelligence. This includes things like:

* **Learning:** Acquiring information and rules for using it.
* **Reasoning:** Using logic and problem-solving skills.
* **Problem-solving:** Identifying and resolving issues.
* **Understanding:** Interpreting information and meaning.
* **Perception:** Recognizing and interpreting sensory input (like images or sounds).
* **Adaptation:** Adjusting to new situations and changing environments.

**2. Different Types of AI**

It's helpful to think about AI in different ways:

* **Narrow AI (Weak AI):** This is the type of AI we see most often today. It's designed to perform a *specific* t

In [21]:
# Save Answer Comparison Results

answers_df = pd.DataFrame(answers_data)

answers_df.to_csv(
    "../dataset/home_answers.csv",
    index=False
)

print("Home Activity Answers Saved Successfully!")
print("Total Questions:", len(answers_df))

Home Activity Answers Saved Successfully!
Total Questions: 15


In [22]:
import os

files = [
    "../dataset/queries.csv",
    "../dataset/retrieved_chunks.csv",
    "../dataset/home_answers.csv"
]

for file in files:
    print(file, "Exists:", os.path.exists(file))

print("\nQueries:", len(queries_df))
print("Retrieved Chunks:", len(retrieved_df))
print("Compared Answers:", len(answers_df))

../dataset/queries.csv Exists: True
../dataset/retrieved_chunks.csv Exists: True
../dataset/home_answers.csv Exists: True

Queries: 15
Retrieved Chunks: 75
Compared Answers: 15


## Observations

1. The PDF contains 17 pages of Artificial Intelligence learning materials.
2. Text was successfully extracted from the PDF.
3. Different chunk sizes were tested:
   - Chunk Size 300: 92 chunks
   - Chunk Size 500: 57 chunks
   - Chunk Size 800: 35 chunks
4. Chunk size 500 with overlap 100 was selected for document retrieval.
5. Sentence Transformers generated embeddings successfully.
6. ChromaDB stored 57 document chunks.
7. Top-5 relevant chunks were retrieved for each of the 15 questions.
8. Answers were generated using Ollama with and without RAG.
9. Both answers were saved for comparison in CSV format.
10. RAG helps generate answers based on the information available in the PDF.